# 제조 품질 이상 탐지 | 최종 모델링 레퍼런스

공식 대회 CSV를 `data/`에 둔 뒤 위에서 아래로 실행하는 공개용 레퍼런스입니다. 원본 실험의 핵심 전처리·샘플링·블렌딩 결정을 재구성했으며, 탐색 셀·중간 출력·수동 예측값 변경은 제외했습니다.

- 입력: `data/train.csv`, `data/test.csv`, `data/submission.csv`
- 검증: fold별 train 데이터에서 전처리 규칙을 학습하는 Stratified 5-fold CV의 F1
- 모델: ROS CatBoost(0.5) + under-2 CatBoost(0.3) + under-3 RandomForest(0.2) 가중 블렌딩
- 출력: `submission_portfolio.csv`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

DATA_DIR = next((path for path in (Path('data'), Path('../data')) if path.exists()), Path('data'))
RANDOM_STATE = 736665
N_SPLITS = 5

## 1. 데이터 로드

대회 원본 CSV는 공개 저장소에 포함하지 않습니다. `test.csv`의 비어 있는 `target` 열은 자동으로 제외합니다.

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
submission = pd.read_csv(DATA_DIR / 'submission.csv')

target = train['target'].map({'Normal': 0, 'AbNormal': 1}).astype(int)
# `Set ID`는 관측치를 식별하는 값이므로 모델 입력에서 제외한다.
feature_drop_columns = ['target', 'Set ID']
train_features = train.drop(columns=feature_drop_columns, errors='ignore')
test_features = test.drop(columns=feature_drop_columns, errors='ignore')

print(f'train: {train_features.shape}, test: {test_features.shape}')
print(target.value_counts(normalize=True).rename({0: 'Normal', 1: 'AbNormal'}))

## 2. 공정 변수 정리와 파생 변수

완전 결측·상수·중복 열을 제거합니다. 각 CV fold에서는 train 부분으로만 제거·상관관계·인코딩 규칙을 학습해 validation 부분에 적용합니다. 최종 제출 단계에서는 전체 train으로 같은 규칙을 다시 학습합니다.

In [ ]:
def clean_ok_coordinates(train_df, test_df):
    """원본 실험에서 `OK`가 섞인 Stage1 X 좌표를 좌표 관계로 보정한다."""
    rules = {
        '_Dam': np.array([550.3, 162.4, 549.0, 549.5, 550.0, 548.5]),
        '_Fill1': np.array([838.4, 837.7, 837.9, 838.2, 837.5]),
        '_Fill2': np.array([835.5, 305.0]),
    }
    for suffix, candidates in rules.items():
        x_col = f'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result{suffix}'
        y_col = f'HEAD NORMAL COORDINATE Y AXIS(Stage1) Collect Result{suffix}'
        z_col = f'HEAD NORMAL COORDINATE Z AXIS(Stage1) Collect Result{suffix}'
        if not {x_col, y_col, z_col}.issubset(train_df.columns) or not {x_col, y_col, z_col}.issubset(test_df.columns):
            continue
        for df in (train_df, test_df):
            for col in (x_col, y_col, z_col):
                df[col] = pd.to_numeric(df[col].replace('OK', np.nan), errors='coerce')
        observed = train_df[[x_col, y_col, z_col]].dropna()
        coordinate_sum = (observed[x_col] + observed[y_col] + observed[z_col]).mean()
        for df in (train_df, test_df):
            missing = df[x_col].isna() & df[y_col].notna() & df[z_col].notna()
            estimated = coordinate_sum - df.loc[missing, y_col] - df.loc[missing, z_col]
            df.loc[missing, x_col] = [candidates[np.abs(candidates - value).argmin()] for value in estimated]
    drop_col = 'HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Dam'
    return train_df.drop(columns=drop_col, errors='ignore'), test_df.drop(columns=drop_col, errors='ignore')


def remove_noninformative_columns(train_df, test_df):
    drop_cols = [
        col for col in train_df.columns
        if train_df[col].isna().all() or train_df[col].nunique(dropna=False) <= 1
    ]

    seen, duplicates = {}, []
    for col in train_df.columns.difference(drop_cols):
        fingerprint = pd.util.hash_pandas_object(train_df[col], index=False).values.tobytes()
        if fingerprint in seen:
            duplicates.append(col)
        else:
            seen[fingerprint] = col

    drop_cols = sorted(set(drop_cols + duplicates))
    return train_df.drop(columns=drop_cols), test_df.drop(columns=drop_cols), drop_cols


def add_correlated_features(train_df, test_df, threshold=0.99999, max_pairs=12):
    numeric_cols = train_df.select_dtypes(include=np.number).columns
    corr = train_df[numeric_cols].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    pairs = [(row, col) for row, col in zip(*np.where(upper.to_numpy() >= threshold))]

    for row, col in pairs[:max_pairs]:
        left, right = upper.index[row], upper.columns[col]
        name = f'{left}__{right}'
        train_df[f'SUM__{name}'] = train_df[left].fillna(0) + train_df[right].fillna(0)
        test_df[f'SUM__{name}'] = test_df[left].fillna(0) + test_df[right].fillna(0)
        denominator_train = train_df[left].replace(0, np.nan)
        denominator_test = test_df[left].replace(0, np.nan)
        train_df[f'DELTA__{name}'] = ((train_df[right] - train_df[left]) / denominator_train).fillna(0)
        test_df[f'DELTA__{name}'] = ((test_df[right] - test_df[left]) / denominator_test).fillna(0)

    return train_df, test_df, len(pairs[:max_pairs])


def prepare_features(fit_df, transform_df):
    """Fit preprocessing rules on fit_df and apply the same rules to transform_df."""
    fit_df, transform_df = clean_ok_coordinates(fit_df.copy(), transform_df.copy())
    fit_df, transform_df, _ = remove_noninformative_columns(fit_df, transform_df)
    fit_df, transform_df, _ = add_correlated_features(fit_df, transform_df)
    return encode_features(fit_df, transform_df)

In [ ]:
def encode_features(train_df, test_df):
    """원본 실험의 label/one-hot 인코딩 구성을 공개용으로 정리한다."""
    train_df, test_df = train_df.copy(), test_df.copy()
    label_columns = [
        'Equipment_Dam', 'Equipment_Fill1', 'Equipment_Fill2',
        'Chamber Temp. Judge Value_AutoClave',
    ]
    one_hot_columns = ['Model.Suffix_Dam', 'Workorder_Dam']

    for col in label_columns:
        if col not in train_df.columns or col not in test_df.columns:
            continue
        encoder = LabelEncoder()
        train_values = train_df[col].fillna('MISSING').astype(str)
        test_values = test_df[col].fillna('MISSING').astype(str)
        train_df[col] = encoder.fit_transform(train_values)
        known = set(encoder.classes_)
        test_df[col] = test_values.map(
            lambda value: encoder.transform([value])[0] if value in known else len(encoder.classes_)
        )

    for col in one_hot_columns:
        if col not in train_df.columns or col not in test_df.columns:
            continue
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        train_encoded = encoder.fit_transform(train_df[[col]].fillna('MISSING'))
        test_encoded = encoder.transform(test_df[[col]].fillna('MISSING'))
        encoded_columns = encoder.get_feature_names_out([col])
        train_df = pd.concat([train_df.drop(columns=col), pd.DataFrame(train_encoded, columns=encoded_columns, index=train_df.index)], axis=1)
        test_df = pd.concat([test_df.drop(columns=col), pd.DataFrame(test_encoded, columns=encoded_columns, index=test_df.index)], axis=1)

    return train_df, test_df

# `prepare_features`는 다음 CV cell에서 fold별로 호출한다.

## 3. Stratified 5-fold 검증과 가중 블렌딩

각 fold의 train 부분에서만 전처리 규칙을 학습한 뒤, ROS CatBoost, under-2 CatBoost, under-3 RandomForest의 확률을 `0.5 / 0.3 / 0.2`로 가중 합산합니다.

In [ ]:
def build_models(seed):
    return (
        CatBoostClassifier(verbose=False, random_state=seed),
        CatBoostClassifier(verbose=False, random_state=seed),
        RandomForestClassifier(random_state=seed),
    )

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof_proba = np.zeros(len(train_features))

for fold, (train_idx, valid_idx) in enumerate(skf.split(train_features, target), start=1):
    X_train, X_valid = prepare_features(
        train_features.iloc[train_idx], train_features.iloc[valid_idx]
    )
    y_train, y_valid = target.iloc[train_idx], target.iloc[valid_idx]
    cat_ros, cat_under_2, rf_under_3 = build_models(RANDOM_STATE + fold)
    X_ros, y_ros = RandomOverSampler(random_state=RANDOM_STATE + fold).fit_resample(X_train, y_train)
    X_under_2, y_under_2 = RandomUnderSampler(sampling_strategy=0.5, random_state=RANDOM_STATE + fold).fit_resample(X_train, y_train)
    X_under_3, y_under_3 = RandomUnderSampler(sampling_strategy=1 / 3, random_state=RANDOM_STATE + fold).fit_resample(X_train, y_train)
    cat_ros.fit(X_ros, y_ros)
    cat_under_2.fit(X_under_2, y_under_2)
    rf_under_3.fit(X_under_3, y_under_3)
    oof_proba[valid_idx] = (
        0.5 * cat_ros.predict_proba(X_valid)[:, 1]
        + 0.3 * cat_under_2.predict_proba(X_valid)[:, 1]
        + 0.2 * rf_under_3.predict_proba(X_valid)[:, 1]
    )

thresholds = np.arange(0.40, 0.60, 0.0001)
scores = [f1_score(target, oof_proba >= threshold) for threshold in thresholds]
best_threshold = float(thresholds[int(np.argmax(scores))])
best_f1 = float(np.max(scores))
print(f'OOF F1: {best_f1:.6f}, threshold: {best_threshold:.4f}')

## 4. 전체 학습과 제출 파일

CV가 끝난 뒤 전체 train으로 전처리 규칙을 다시 학습하고 test에 적용합니다. 임의의 행을 수동으로 변경하지 않습니다.

In [ ]:
X_full, X_test = prepare_features(train_features, test_features)
print(f'final encoded features: {X_full.shape[1]}')

cat_ros, cat_under_2, rf_under_3 = build_models(RANDOM_STATE)
X_ros, y_ros = RandomOverSampler(random_state=RANDOM_STATE).fit_resample(X_full, target)
X_under_2, y_under_2 = RandomUnderSampler(sampling_strategy=0.5, random_state=RANDOM_STATE).fit_resample(X_full, target)
X_under_3, y_under_3 = RandomUnderSampler(sampling_strategy=1 / 3, random_state=RANDOM_STATE).fit_resample(X_full, target)

cat_ros.fit(X_ros, y_ros)
cat_under_2.fit(X_under_2, y_under_2)
rf_under_3.fit(X_under_3, y_under_3)

test_proba = (
    0.5 * cat_ros.predict_proba(X_test)[:, 1]
    + 0.3 * cat_under_2.predict_proba(X_test)[:, 1]
    + 0.2 * rf_under_3.predict_proba(X_test)[:, 1]
)
submission['target'] = np.where(test_proba >= best_threshold, 'AbNormal', 'Normal')
submission.to_csv('submission_portfolio.csv', index=False)
submission.head()